# NOVA — MJX walking-policy training (Colab GPU)

Trains the calibrated NOVA quadruped locomotion policy (Brax PPO on the MJX env).
The actuator model is grounded in the measured STS3215 (velocity cap, deadband,
backlash, latency — see `docs/bench/README.md`).

**Before running:** `Runtime -> Change runtime type -> GPU` (T4 is enough).

Flow: GPU check -> get code -> install pinned deps -> sanity (STANDS) ->
mount Drive -> **train** (resumable) -> rollout video -> export for deploy.

Checkpoints live on Google Drive, so a Colab disconnect costs nothing — just
re-run the **Train** cell and it resumes from the latest checkpoint.

## 1. GPU + Python check

In [ ]:
import sys
print('Python', sys.version.split()[0])   # want 3.11 or 3.12 (jax 0.6.0 window)
!nvidia-smi -L || echo 'NO GPU -> Runtime > Change runtime type > GPU'

## 2. Get the code
Public repo — plain shallow clone, no token needed.

_Already cloned this session?_ Skip the clone and just pull the latest:
`%cd /content/LE_NOVA` → `!git pull` → `%cd sim/nova_mjx`.

In [ ]:
!git clone --depth 1 https://github.com/Ace2932/LE_NOVA.git 2>&1 | tail -2
%cd LE_NOVA/sim/nova_mjx
!ls

## 3. Install pinned deps
`brax 0.14.2` + `jax 0.6.0` is the validated window (see `requirements.txt`).

⚠ If the **sanity** cell below reports `backend cpu`, the pre-installed jax
wasn't fully replaced: `Runtime -> Restart session`, then re-run from **this**
cell (the clone persists, so you can skip cell 2).

In [ ]:
!pip install -q "jax[cuda12]==0.6.0" brax==0.14.2 "orbax-checkpoint>=0.11.22" \
    "mujoco>=3.10" "mujoco-mjx>=3.10" "imageio>=2.31" imageio-ffmpeg 2>&1 | tail -3
print('deps installed — if the next cell says cpu, restart runtime + re-run from here')

## 4. Sanity — GPU visible + model STANDS

In [ ]:
import jax
print('jax backend:', jax.default_backend(), '| devices:', jax.devices())
assert jax.default_backend() == 'gpu', 'not on GPU — restart runtime + re-run cell 3'
!python build_mjcf.py && python validate_model.py

## 5. Mount Drive (persistent checkpoints)
Checkpoints go here so a disconnect never loses progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/nova_ckpt'
import os; os.makedirs(CKPT, exist_ok=True); print('checkpoints ->', CKPT)

## 6. TRAIN  (resumable)
~60M steps ~= a solid flat-ground gait on a T4 (roughly 30-60 min). Watch
`eval_reward` climb. **If Colab disconnects, just re-run this cell** — it finds
the latest checkpoint on Drive and continues. Re-run again to train longer.

In [ ]:
!python train.py --ckpt $CKPT --timesteps 60_000_000 --num_envs 2048

## 7. Rollout — watch it walk
Renders the trained policy at a fixed forward command. If the gait looks wrong,
train longer (re-run cell 6) or tune rewards in `env.py`.

In [ ]:
!python rollout.py --policy nova_policy.pkl --vx 0.5 --steps 400 --out walk.mp4
from IPython.display import Video
Video('walk.mp4', embed=True, width=480)

## 8. Export for deploy
Writes `nova_policy.npz` (framework-free, what `policy_runner`/`policy_node`
load on the Jetson) + `nova_policy.onnx`, and copies them to Drive.

In [ ]:
!python export_policy.py --policy nova_policy.pkl
!cp -v nova_policy.npz nova_policy.onnx $CKPT/ 2>/dev/null; echo 'exported to Drive'